## This notebook is for implementing the matching algorithm for ThirdSpace


In [82]:
import pandas as pd
import ast

In [102]:
today = pd.to_datetime("today").date()
venues = pd.read_csv("../data/ThirdSpaceVenues.csv")
users = pd.read_csv("../data/User_Responses_Cleaned.csv")
streaming = pd.read_csv(f"../data/streaming_cleaned_{today}.csv")

In [103]:
venues.head()

,Name,Link,Vibe,StreamingServices
0,Alley Bar,https://www.a2alleybar.com,Signature Cocktails & Spirits,Cable
1,Ashley's Restaurant,https://ashleys.com/?srsltid=AfmBOopoXlA_38WIv...,Traditional Sports Bar Environment (Multiple s...,Cable
2,Babs' Underground,https://www.babsannarbor.com,Signature Cocktails & Spirits,Cable
3,Chapala Mexican Restaurant,https://eatchapala.com,High-Quality Food Menu (Beyond standard pub fa...,"Cable, Peacock"
4,Detroit Pizza Pub,https://detroitpizzapub.com,High-Quality Food Menu (Beyond standard pub fa...,Cable


In [104]:
users.head()

,timestamp,consent,fullname,email,venueFeatures,MCBB,WCBB,NBA,additionalComments,venueFeatureVector
0,3/6/2026 11:23:36,"Yes, I consent to participate and proceed",Test 1,lumandigo6@gmail.com,Traditional Sports Bar Environment (Multiple s...,University of North Carolina; University of Mi...,University of North Carolina,Atlanta Hawks; Detroit Pistons,NaN,"[1, 1, 0, 0, 0]"
1,3/6/2026 11:25:08,"Yes, I consent to participate and proceed",Test 2,lundenm@umich.edu,Traditional Sports Bar Environment (Multiple s...,University of Michigan; University of North Ca...,University of Iowa,Detroit Pistons; Charlotte Hornets,NaN,"[1, 1, 0, 1, 0]"
2,3/6/2026 11:26:03,"Yes, I consent to participate and proceed",Test 3,lumandigo6@gmail.com,"Quality Beer Selection, Signature Cocktails & ...",Duke University; University of Georgia,NaN,Los Angeles Lakers,NaN,"[0, 1, 1, 0, 0]"
3,3/6/2026 11:27:20,"Yes, I consent to participate and proceed",Test 4,lundenm@umich.edu,Family-Friendly Atmosphere,Indiana University; Notre Dame,Indiana University; Notre Dame,Indiana Pacers,NaN,"[0, 0, 0, 0, 1]"
4,3/6/2026 11:28:37,"Yes, I consent to participate and proceed",Test 5,lumandigo6@gmail.com,"Quality Beer Selection, High-Quality Food Menu...",University of Arizona; University of North Car...,NaN,Miami Heat; Los Angeles Lakers,NaN,"[0, 1, 0, 1, 0]"


In [105]:
streaming.sample(10)

,Date,Time,League,Matchup,Services,DayOfWeek
49,2026-03-12,15:00,WCBB,"CSU Fullerton,Hawai'i","['ESPN Select', 'ESPN Unlimited']",Thursday
23,2026-03-12,18:30,MCBB,"Northwestern,Purdue",['Big Ten Network'],Thursday
52,2026-03-12,17:00,WCBB,"Stephen F. Austin,McNeese","['Fubo Sports', 'ESPN Unlimited', 'ESPN U', 'E...",Thursday
24,2026-03-12,18:30,MCBB,"Ohio,Kent State","['ESPN Select', 'ESPN Unlimited']",Thursday
21,2026-03-12,18:00,MCBB,"South Carolina State,Norfolk State","['ESPN Select', 'ESPN Unlimited']",Thursday
57,2026-03-12,19:00,WCBB,"Evansville,UIC","['ESPN Select', 'ESPN Unlimited']",Thursday
45,2026-03-12,23:30,MCBB,"UC Davis,CSU Fullerton","['ESPN Select', 'ESPN Unlimited']",Thursday
47,2026-03-12,15:00,WCBB,"Tarleton State,Utah Valley","['ESPN Select', 'ESPN Unlimited']",Thursday
56,2026-03-12,19:00,WCBB,"Le Moyne,Fairleigh Dickinson","['ESPN Select', 'ESPN Unlimited']",Thursday
25,2026-03-12,19:00,MCBB,"Florida State,Duke","['Fubo Sports', 'ESPN Unlimited', 'ESPN']",Thursday


### Create fanbase sizes

### Add fanbase preferences of venue

In [106]:
from collections import Counter, defaultdict
import pandas as pd

# 1. Set up a nested dictionary. 
# For each team, we will track its total "size" and a Counter of its "features"
fanbase_stats = {
    "NBA": defaultdict(lambda: {"size": 0, "features": Counter()}),
    "MCBB": defaultdict(lambda: {"size": 0, "features": Counter()}),
    "WCBB": defaultdict(lambda: {"size": 0, "features": Counter()})
}

# 2. Loop through each row in the dataframe
for index, row in users.iterrows():
    
    # Safely grab and split the venue features for this specific user
    venue_str = row['venueFeatures']
    user_features = []
    if pd.notna(venue_str):
        # Splitting by comma and stripping extra spaces
        user_features = [feature.strip() for feature in venue_str.split(",")]

    # 3. Loop through each sport
    for sport in fanbase_stats.keys():
        teams_str = row[sport]
        
        # Safely grab and split the teams
        if pd.notna(teams_str):
            user_teams = teams_str.split("; ")
            
            for team in user_teams:
                # Increment the total fanbase size
                fanbase_stats[sport][team]["size"] += 1
                
                # Increment the count for every venue feature this user selected
                for feature in user_features:
                    fanbase_stats[sport][team]["features"][feature] += 1

flat_data = []
for sport, teams in fanbase_stats.items():
    for team, stats in teams.items():
        most_common_feature = stats['features'].most_common(1)[0][0] if stats['features'] else None
        flat_data.append({
            "Sport": sport,
            "Team": team,
            "Fanbase_Size": stats["size"],
            "Most_Common_Venue_Feature": most_common_feature
        })
fanbase_df = pd.DataFrame(flat_data).sort_values(by="Fanbase_Size", ascending=False).reset_index(drop=True)
fanbase_df.head()

,Sport,Team,Fanbase_Size,Most_Common_Venue_Feature
0,NBA,Detroit Pistons,5,Traditional Sports Bar Environment (Multiple s...
1,MCBB,University of Michigan,4,Traditional Sports Bar Environment (Multiple s...
2,NBA,Los Angeles Lakers,3,Quality Beer Selection
3,MCBB,University of North Carolina,3,Quality Beer Selection
4,WCBB,University of Michigan,2,Traditional Sports Bar Environment (Multiple s...


In [108]:
display(streaming.head())
display(venues.head())

,Date,Time,League,Matchup,Services,DayOfWeek
0,2026-03-12,19:00,NBA,"Detroit Pistons,Philadelphia 76ers",['Prime Video'],Thursday
1,2026-03-12,19:00,NBA,"Indiana Pacers,Phoenix Suns",['NBA League Pass'],Thursday
2,2026-03-12,19:00,NBA,"Orlando Magic,Washington Wizards",['NBA League Pass'],Thursday
3,2026-03-12,19:30,NBA,"Atlanta Hawks,Brooklyn Nets",['NBA League Pass'],Thursday
4,2026-03-12,19:30,NBA,"Miami Heat,Milwaukee Bucks",['NBA League Pass'],Thursday


,Name,Link,Vibe,StreamingServices
0,Alley Bar,https://www.a2alleybar.com,Signature Cocktails & Spirits,Cable
1,Ashley's Restaurant,https://ashleys.com/?srsltid=AfmBOopoXlA_38WIv...,Traditional Sports Bar Environment (Multiple s...,Cable
2,Babs' Underground,https://www.babsannarbor.com,Signature Cocktails & Spirits,Cable
3,Chapala Mexican Restaurant,https://eatchapala.com,High-Quality Food Menu (Beyond standard pub fa...,"Cable, Peacock"
4,Detroit Pizza Pub,https://detroitpizzapub.com,High-Quality Food Menu (Beyond standard pub fa...,Cable


In [109]:
cable_channels = set(['ABC', 'CBS', 'NBC', 'Telemundo', 'USA Network',
        'ESPN', 'ESPN2', 'ESPN U', 'ESPNews', 'ESPN App',
        'CBS Sports Network', 'Big Ten Network', 'SEC Network', 'ACC Network',
        'NBA TV', 'NBC Sports Network', # NBCSN is defunct, but was national
        
        # Local to Detroit / Michigan
        'FanDuel Sports Network Detroit Extra',
        'Detroit SportsNet', 'Youtube TV'])

In [110]:
Sport_Team_lists = [row[["Sport", "Team", "Most_Common_Venue_Feature"]].tolist() for _, row in fanbase_df.iterrows()]


for sport, team, venue_feature in Sport_Team_lists:
    # locate match in streaming schedule for the sport and team
    matching_games = streaming[(streaming['League'] == sport) & (streaming['Matchup'].str.lower().str.contains(team.lower(), na=False))]
    if not matching_games.empty:
        print(f"Matches for {sport} - {team}:")

Matches for NBA - Detroit Pistons:
Matches for NBA - Los Angeles Lakers:
Matches for NBA - Atlanta Hawks:
Matches for MCBB - University of Georgia:
Matches for NBA - San Antonio Spurs:
Matches for NBA - Miami Heat:
Matches for NBA - Indiana Pacers:


In [112]:
import pandas as pd
import re

# 1. Create a helper function to strip formal academic words
def simplify_team_name(name):
    # Remove "University of " or "The University of " from the start
    name = re.sub(r'(?i)^(the\s+)?university\s+of\s+', '', name)
    # Remove " University" or " College" from the end
    name = re.sub(r'(?i)\s+university$', '', name)
    name = re.sub(r'(?i)\s+college$', '', name)
    # Add any other specific weird ones you find here!
    
    return name.strip()

Sport_Team_lists = [row[["Sport", "Team", "Most_Common_Venue_Feature"]].tolist() for _, row in fanbase_df.iterrows()]

for sport, team, venue_feature in Sport_Team_lists:
    
    # 2. Pass the user's team through the cleaner
    search_team = simplify_team_name(team)
    
    # 3. Search using the simplified name
    matching_games = streaming[
        (streaming['League'] == sport) & 
        (streaming['Matchup'].str.lower().str.contains(search_team.lower(), regex=False, na=False))
    ]
    
    if not matching_games.empty:
        print(f"Matches for {sport} - {team} (Searched as '{search_team}'):")

Matches for NBA - Detroit Pistons (Searched as 'Detroit Pistons'):
Matches for NBA - Los Angeles Lakers (Searched as 'Los Angeles Lakers'):
Matches for MCBB - University of North Carolina (Searched as 'North Carolina'):
Matches for NBA - Atlanta Hawks (Searched as 'Atlanta Hawks'):
Matches for MCBB - Purdue University (Searched as 'Purdue'):
Matches for WCBB - Indiana University (Searched as 'Indiana'):
Matches for MCBB - University of Arizona (Searched as 'Arizona'):
Matches for MCBB - University of Georgia (Searched as 'Georgia'):
Matches for MCBB - Duke University (Searched as 'Duke'):
Matches for NBA - San Antonio Spurs (Searched as 'San Antonio Spurs'):
Matches for NBA - Miami Heat (Searched as 'Miami Heat'):
Matches for NBA - Indiana Pacers (Searched as 'Indiana Pacers'):


In [99]:
import pandas as pd
import ast

Sport_Team_lists = [row[["Sport", "Team", "Most_Common_Venue_Feature"]].tolist() for _, row in fanbase_df.iterrows()]

games_venues_mappings = []
for sport, team, venue_feature in Sport_Team_lists:
    
    # locate match in streaming schedule for the sport and team
    matching_games = streaming[
        (streaming['League'] == sport) & 
        (streaming['Matchup'].str.lower().str.contains(team.lower(), na=False))
    ]
    
    if not matching_games.empty:
        game_venue_mapping = {
            "Sport": sport,
            "Team": team,
            "Game": matching_games.iloc[0]['Matchup'],
            "Time": matching_games.iloc[0]['Time'],
            "Date": matching_games.iloc[0]['Date'],
            "Services": ast.literal_eval(matching_games.iloc[0]['Services']),
            "Venue_Feature": venue_feature
        }

        # Assuming 'cable_channels' is a list defined earlier in your script
        game_venue_mapping['Services'] = [
            "Cable" if service.strip() in cable_channels else service 
            for service in game_venue_mapping['Services']
        ]

        # 1. Find venues that match the vibe
        relevant_venues = venues[
            venues['Vibe'].str.lower().str.contains(venue_feature.lower(), regex=False, na=False)
        ]
        
        # 2. Filter venues by streaming service using .split(',') instead of ast.literal_eval
        relevant_venues = relevant_venues[
            relevant_venues['StreamingServices'].apply(
                lambda x: any(service.strip() in [s.strip() for s in str(x).split(',')] for service in game_venue_mapping['Services']) if pd.notna(x) else False
            )
        ]

        # 3. Grab the 'Name' column instead of 'Venue'
        game_venue_mapping['Recommended_Venue'] = relevant_venues.iloc[0]['Name'] if not relevant_venues.empty else "No matching venue found"
        
        games_venues_mappings.append(game_venue_mapping)

# Convert your final mappings back into a dataframe to view
final_mappings_df = pd.DataFrame(games_venues_mappings)

In [115]:
import pandas as pd
import ast
import re

# 1. Helper function to strip formal academic words for better matching
def simplify_team_name(name):
    name = re.sub(r'(?i)^(the\s+)?university\s+of\s+', '', name)
    name = re.sub(r'(?i)\s+university$', '', name)
    name = re.sub(r'(?i)\s+college$', '', name)
    return name.strip()

Sport_Team_lists = [row[["Sport", "Team", "Most_Common_Venue_Feature"]].tolist() for _, row in fanbase_df.iterrows()]

games_venues_mappings = []
for sport, team, venue_feature in Sport_Team_lists:
    
    # 2. Pass the user's team through the cleaner
    search_team = simplify_team_name(team)
    
    # THE FIX: Create an exact-match regex pattern. 
    # This looks for: (Start of string OR a comma) + the team name + (End of string OR a comma)
    exact_match_pattern = rf"(?:^|,)\s*{re.escape(search_team.lower())}\s*(?:,|$)"
    
    # 3. Locate match using the new exact boundary pattern
    matching_games = streaming[
        (streaming['League'] == sport) & 
        (streaming['Matchup'].str.lower().str.contains(exact_match_pattern, regex=True, na=False))
    ]
    
    if not matching_games.empty:
        game_venue_mapping = {
            "Sport": sport,
            "Team": team, 
            "Game": matching_games.iloc[0]['Matchup'],
            "Time": matching_games.iloc[0]['Time'],
            "Date": matching_games.iloc[0]['Date'],
            "Services": ast.literal_eval(matching_games.iloc[0]['Services']),
            "Venue_Feature": venue_feature
        }

        # Assuming 'cable_channels' is a list defined earlier
        game_venue_mapping['Services'] = [
            "Cable" if service.strip() in cable_channels else service 
            for service in game_venue_mapping['Services']
        ]

        # 1. Find venues that match the vibe
        relevant_venues = venues[
            venues['Vibe'].str.lower().str.contains(venue_feature.lower(), regex=False, na=False)
        ]
        
        # 2. Filter venues by streaming service
        relevant_venues = relevant_venues[
            relevant_venues['StreamingServices'].apply(
                lambda x: any(service.strip() in [s.strip() for s in str(x).split(',')] for service in game_venue_mapping['Services']) if pd.notna(x) else False
            )
        ]

        # 3. Grab the 'Name' column
        game_venue_mapping['Recommended_Venue'] = relevant_venues.iloc[0]['Name'] if not relevant_venues.empty else "No matching venue found"
        
        games_venues_mappings.append(game_venue_mapping)

# Convert your final mappings back into a dataframe to view
final_mappings_df = pd.DataFrame(games_venues_mappings)

In [117]:
import pandas as pd
import ast
import re

# Helper function to strip formal academic words for better matching
def simplify_team_name(name):
    name = re.sub(r'(?i)^(the\s+)?university\s+of\s+', '', name)
    name = re.sub(r'(?i)\s+university$', '', name)
    name = re.sub(r'(?i)\s+college$', '', name)
    return name.strip()

Sport_Team_lists = [row[["Sport", "Team", "Most_Common_Venue_Feature"]].tolist() for _, row in fanbase_df.iterrows()]

games_venues_mappings = []

# NEW: Keep track of venues we've already used
assigned_venues = set()

for sport, team, venue_feature in Sport_Team_lists:
    
    search_team = simplify_team_name(team)
    exact_match_pattern = rf"(?:^|,)\s*{re.escape(search_team.lower())}\s*(?:,|$)"
    
    matching_games = streaming[
        (streaming['League'] == sport) & 
        (streaming['Matchup'].str.lower().str.contains(exact_match_pattern, regex=True, na=False))
    ]
    
    if not matching_games.empty:
        game_venue_mapping = {
            "Sport": sport,
            "Team": team, 
            "Game": matching_games.iloc[0]['Matchup'],
            "Time": matching_games.iloc[0]['Time'],
            "Date": matching_games.iloc[0]['Date'],
            "Services": ast.literal_eval(matching_games.iloc[0]['Services']),
            "Venue_Feature": venue_feature
        }

        game_venue_mapping['Services'] = [
            "Cable" if service.strip() in cable_channels else service 
            for service in game_venue_mapping['Services']
        ]

        # Helper function to check if a venue has the required streaming services
        def has_streaming(venue_services_str):
            if pd.isna(venue_services_str): 
                return False
            venue_services = [s.strip() for s in str(venue_services_str).split(',')]
            return any(service.strip() in venue_services for service in game_venue_mapping['Services'])

        # 1. Filter out venues that have already been assigned to another game
        available_venues = venues[~venues['Name'].isin(assigned_venues)]
        
        # 2. Look for the "Perfect Match" (Matches Vibe AND Streaming)
        vibe_match = available_venues[
            available_venues['Vibe'].str.lower().str.contains(venue_feature.lower(), regex=False, na=False)
        ]
        perfect_matches = vibe_match[vibe_match['StreamingServices'].apply(has_streaming)]

        # 3. Decision Logic: Assign venue and record it so it can't be used again
        if not perfect_matches.empty:
            selected_venue = perfect_matches.iloc[0]['Name']
        else:
            # FALLBACK: Ignore the vibe, just find ANY available venue with the right streaming service
            fallback_matches = available_venues[available_venues['StreamingServices'].apply(has_streaming)]
            if not fallback_matches.empty:
                selected_venue = fallback_matches.iloc[0]['Name']
            else:
                # If we run out of venues, or none have the stream
                selected_venue = "No matching venue found"
        
        game_venue_mapping['Recommended_Venue'] = selected_venue
        
        # If we successfully found a venue, add it to our tracking set
        if selected_venue != "No matching venue found":
            assigned_venues.add(selected_venue)
            
        games_venues_mappings.append(game_venue_mapping)

final_mappings_df = pd.DataFrame(games_venues_mappings)

In [118]:
final_mappings_df

,Sport,Team,Game,Time,Date,Services,Venue_Feature,Recommended_Venue
0,NBA,Detroit Pistons,"Detroit Pistons,Philadelphia 76ers",19:00,2026-03-12,[Prime Video],Traditional Sports Bar Environment (Multiple s...,The Grotto - Watering Hole
1,NBA,Los Angeles Lakers,"Los Angeles Lakers,Chicago Bulls",22:30,2026-03-12,[NBA League Pass],Quality Beer Selection,No matching venue found
2,MCBB,University of North Carolina,"Clemson,North Carolina",21:30,2026-03-12,"[Fubo Sports, ESPN Unlimited, Cable]",Quality Beer Selection,Ashley's Restaurant
3,NBA,Atlanta Hawks,"Atlanta Hawks,Brooklyn Nets",19:30,2026-03-12,[NBA League Pass],Traditional Sports Bar Environment (Multiple s...,No matching venue found
4,MCBB,Purdue University,"Northwestern,Purdue",18:30,2026-03-12,[Cable],Wings,Alley Bar
5,MCBB,University of Arizona,"UCF,Arizona",15:00,2026-03-12,"[Fubo Sports, ESPN Unlimited, Cable]",Quality Beer Selection,Grizzly Peak Brewing Company
6,MCBB,Duke University,"Florida State,Duke",19:00,2026-03-12,"[Fubo Sports, ESPN Unlimited, Cable]",Quality Beer Selection,Heidelberg Restaurant
7,NBA,San Antonio Spurs,"San Antonio Spurs,Denver Nuggets",21:00,2026-03-12,[NBA League Pass],Traditional Sports Bar Environment (Multiple s...,No matching venue found
8,NBA,Miami Heat,"Miami Heat,Milwaukee Bucks",19:30,2026-03-12,[NBA League Pass],Quality Beer Selection,No matching venue found
9,NBA,Indiana Pacers,"Indiana Pacers,Phoenix Suns",19:00,2026-03-12,[NBA League Pass],Family-Friendly Atmosphere,No matching venue found


In [ ]:
import numpy as np

# Initialize empty lists to hold the new column data
assigned_venues_list = []
todays_games_list = []
game_times_list = []

for index, row in users.iterrows():
    # Safely extract the user's teams for each sport into clean lists
    user_mcbb = [t.strip() for t in str(row['MCBB']).split(';')] if pd.notna(row['MCBB']) else []
    user_wcbb = [t.strip() for t in str(row['WCBB']).split(';')] if pd.notna(row['WCBB']) else []
    user_nba = [t.strip() for t in str(row['NBA']).split(';')] if pd.notna(row['NBA']) else []
    
    user_venues = []
    user_games = []
    user_times = []
    
    # NEW: A clean way to track duplicates without getting confused by the new sport prefixes
    seen_games = set() 
    
    # Loop through the final mappings to see if any of this user's teams are playing
    for _, game_row in final_mappings_df.iterrows():
        sport = game_row['Sport']
        team = game_row['Team']
        venue = game_row['Recommended_Venue']
        game_matchup = game_row['Game']
        
        # NEW: Skip this game entirely if there is no venue to send the user to
        if venue == "No matching venue found":
            continue
            
        # Check if this specific game matches the user's explicit preferences
        is_match = False
        if sport == 'MCBB' and team in user_mcbb:
            is_match = True
        elif sport == 'WCBB' and team in user_wcbb:
            is_match = True
        elif sport == 'NBA' and team in user_nba:
            is_match = True
            
        if is_match:
            # Add to the user's daily list (checking to avoid duplicate entries)
            if game_matchup not in seen_games: 
                seen_games.add(game_matchup)
                
                # NEW: Format the game string to include the sport!
                user_games.append(f"[{sport}] {game_matchup}")
                user_times.append(game_row['Time'])
                user_venues.append(venue)
    
    # If they have games, join them into a readable string. Otherwise, use NaN.
    if user_games:
        assigned_venues_list.append("; ".join(user_venues))
        todays_games_list.append("; ".join(user_games))
        game_times_list.append("; ".join(user_times))
    else:
        assigned_venues_list.append(np.nan)
        todays_games_list.append(np.nan)
        game_times_list.append(np.nan)

# Assign the new lists as columns in the users dataframe
users['assigned_venues'] = assigned_venues_list
users['todays_games'] = todays_games_list
users['game_times'] = game_times_list

# Display the updated dataframe to verify
print(users[['fullname', 'todays_games', 'game_times', 'assigned_venues']].head(10))

           fullname                                       todays_games  \
0            Test 1  [NBA] Detroit Pistons,Philadelphia 76ers; [MCB...   
1            Test 2  [NBA] Detroit Pistons,Philadelphia 76ers; [MCB...   
2            Test 3                          [MCBB] Florida State,Duke   
3            Test 4                                                NaN   
4            Test 5  [MCBB] Clemson,North Carolina; [MCBB] UCF,Arizona   
5            Test 6  [NBA] Detroit Pistons,Philadelphia 76ers; [MCB...   
6  Tyrone Pettygrue           [NBA] Detroit Pistons,Philadelphia 76ers   
7         Alex Arce           [NBA] Detroit Pistons,Philadelphia 76ers   

     game_times                                    assigned_venues  
0  19:00; 21:30    The Grotto - Watering Hole; Ashley's Restaurant  
1  19:00; 21:30    The Grotto - Watering Hole; Ashley's Restaurant  
2         19:00                              Heidelberg Restaurant  
3           NaN                                          

In [123]:
users.to_csv("../data/User_Responses_With_Venues.csv", index=False)